In [1]:
###################################
# IMPORT
###################################
from polars import col, concat, lit, when
import polars as pl


import inspect
import sys; sys.path.append('.'); 
from scripts.hdb_helpers import sample_hdb
import scripts.hdb_helpers as dc



# SAMPLED = 'on'
SAMPLED = None

In [2]:
###################################
# DATA LOAD
###################################
hdbdata = pl.read_parquet('datalake/hdb/datastore/hdbdata_validate_pass')
hdbdata.glimpse()

Rows: 459002
Columns: 13
$ month               <str> '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01', '2000-01'
$ town                <str> 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO'
$ flat_type           <str> '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM'
$ block               <str> '170', '174', '216', '215', '218', '320', '320', '330', '330', '332'
$ street_name         <str> 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1'
$ storey_range        <str> '07 TO 09', '04 TO 06', '07 TO 09', '07 TO 09', '07 TO 09', '04 TO 06', '07 TO 09', '07 TO 09', '04 TO 06', '07 TO 09'
$ floor_area_sqm      <f64> 69.0, 61.0, 73.0, 73.0, 67.0, 73.0, 73.0, 68.

In [3]:
'''
REQUIREMENT:
You will need to design
an ETL pipeline using datasets from January 2012 to December 2016.
'''

# SCOPE THE ASSET FROM HERE ON TI ONLY BETWEEN JAN 2012 TO DEC 2016
hdbdata_periodscope = hdbdata.filter(col('month')>='2012-01').filter(col('month')<='2016-12')

# validate
hdbdata_periodscope['month'].describe()

statistic,value
str,str
"""count""","""92543"""
"""null_count""","""0"""
"""min""","""2012-01"""
"""max""","""2016-12"""


In [4]:
# User may review the source code for helper function random sampling the hdb dataset
print(inspect.getsource(sample_hdb))

def sample_hdb(hdb_df, N_ROWS, SAMPLE_SEED):
    '''
    For sampling hdb data to N_ROWS

    '''
    # take full data why not
    import polars as pl
    core_cols = [c for c in hdb_df.columns if c != 'remaining_lease']
    dataload_nonulls = hdb_df.drop_nulls(subset=core_cols)

    # sampling portion
    SAMPLE_SEED = 1   # change to 2, 3, 4, 5 for different samples
    N_PER_TABLE= round(N_ROWS/3)

    df_sample = (
        dataload_nonulls
        .group_by('tabling_version', maintain_order=True)
        .map_groups(lambda g: g.sample(n=min(N_PER_TABLE, g.height), seed=SAMPLE_SEED))
    )
    return(df_sample)

    hdbdata = df_sample



In [5]:
#### SAMPLING option FOR FIRST EASE OF DATA CLEANING ###

if SAMPLED == 'on': 
    print('using sampled reduced data')
    hdbdata_v1 = sample_hdb(hdbdata_periodscope, N_ROWS=15, SAMPLE_SEED=4)
else:
    print('using full data')
    hdbdata_v1 = hdbdata_periodscope

###################################################

hdbdata_v1.glimpse()

using full data
Rows: 92543
Columns: 13
$ month               <str> '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01', '2012-01'
$ town                <str> 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO', 'ANG MO KIO'
$ flat_type           <str> '2 ROOM', '2 ROOM', '2 ROOM', '2 ROOM', '2 ROOM', '2 ROOM', '3 ROOM', '3 ROOM', '3 ROOM', '3 ROOM'
$ block               <str> '406', '314', '314', '170', '174', '508', '174', '216', '332', '418'
$ street_name         <str> 'ANG MO KIO AVE 10', 'ANG MO KIO AVE 3', 'ANG MO KIO AVE 3', 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 8', 'ANG MO KIO AVE 4', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 1', 'ANG MO KIO AVE 10'
$ storey_range        <str> '01 TO 03', '07 TO 09', '10 TO 12', '01 TO 03', '07 TO 09', '04 TO 06', '07 TO 09', '07 TO 09', '10 TO 12', '04 TO 06'
$ floor_area_sqm      <f64> 44.0, 44.0, 44.0, 45.0, 45.0

In [6]:
'''
REQUIREMENT: Assume HDB lease is 99 years old, recompute remaining lease as of today. Remaining lease
should be rounded down to Years and Months.
Source: description by DATAGOV,
Remaining Lease	-	Remaining time left on the lease of flat.

'''

# Lets take a look at the target column asked to be recomputed
dc.sortcount(hdbdata_v1, 'remaining_lease')
# We assume the reason to recompute remaining lease is, that 
# 1) the value does not exist for 2 tabling versions, and we can recompute a valid information for all tabling versions anyway

92543
shape: (40, 3)
┌─────────────────┬───────┬──────────┐
│ remaining_lease ┆ count ┆ pcnt     │
│ ---             ┆ ---   ┆ ---      │
│ i64             ┆ u32   ┆ f64      │
╞═════════════════╪═══════╪══════════╡
│ null            ┆ 55390 ┆ 0.598533 │
│ 68              ┆ 2328  ┆ 0.025156 │
│ 67              ┆ 1857  ┆ 0.020066 │
│ 69              ┆ 1635  ┆ 0.017667 │
│ 71              ┆ 1547  ┆ 0.016717 │
│ …               ┆ …     ┆ …        │
│ 92              ┆ 346   ┆ 0.003739 │
│ 89              ┆ 308   ┆ 0.003328 │
│ 56              ┆ 296   ┆ 0.003199 │
│ 91              ┆ 294   ┆ 0.003177 │
│ 53              ┆ 270   ┆ 0.002918 │
└─────────────────┴───────┴──────────┘


remaining_lease,count,pcnt
i64,u32,f64
null,55390,0.598533
68,2328,0.025156
67,1857,0.020066
69,1635,0.017667
71,1547,0.016717
…,…,…
50,138,0.001491
96,107,0.001156
49,88,0.000951


In [7]:
'''
# compute: remaining lease as of today, rounded down to years + months
# Since the best information we have which is lease_commence_year, does not have month, 
# but we need to have a sensible proxy of Year and Months of lease remaining, 
# we will make assumption for the lease for all flats commences on 1st January of lease_commence_year
'''

'\n# compute: remaining lease as of today, rounded down to years + months\n# Since the best information we have which is lease_commence_year, does not have month, \n# but we need to have a sensible proxy of Year and Months of lease remaining, \n# we will make assumption for the lease for all flats commences on 1st January of lease_commence_year\n'

In [8]:
# A caution that it represents years! so we should rename it going forward AND keep the revision in afterwards data assets for clarity
hdbdata_v1.select('lease_commence_date').show()

lease_commence_date
i64
1979
1978
1978
1986
1986


In [9]:
hdbdata_v1 = hdbdata_v1.rename({'lease_commence_date': 'lease_commence_year'})

In [10]:
from datetime import datetime
TODAY_YEAR = datetime.now().year
TODAY_MONTH = datetime.now().month
print(TODAY_YEAR)
print(TODAY_MONTH)

2026
7


In [11]:
hdbdata_v1_sel = hdbdata_v1.select('record_id', 'lease_commence_year')

hdbdata_v2 = (
    hdbdata_v1_sel
    .with_columns(lit(TODAY_YEAR).alias('TODAY_YEAR'))
    .with_columns((lit(TODAY_MONTH) + 1).alias('NEXT_MONTH'))
    .with_columns(lease_end_year = col('lease_commence_year') + 99)
    .with_columns(nb_years_left_from_next_year = col('lease_end_year') - (TODAY_YEAR + 1))
    .with_columns(nb_months_to_next_year = 12 - col('NEXT_MONTH'))
)

# initially we decided to do by months, but the logic was not apparent and checkable by external audience quickly.
# Hence we decided to do the below sequence of logic 

In [12]:
# validate
# printed, suggest quick validation, visually computing from left to right, for any chosen rows.
hdbdata_v2.show(1)
hdbdata_v2.show(2)
hdbdata_v2.show(3)

record_id,lease_commence_year,TODAY_YEAR,NEXT_MONTH,lease_end_year,nb_years_left_from_next_year,nb_months_to_next_year
str,i64,i32,i32,i64,i64,i32
"""002-366463""",1979,2026,8,2078,51,4


record_id,lease_commence_year,TODAY_YEAR,NEXT_MONTH,lease_end_year,nb_years_left_from_next_year,nb_months_to_next_year
str,i64,i32,i32,i64,i64,i32
"""002-366463""",1979,2026,8,2078,51,4
"""002-366464""",1978,2026,8,2077,50,4


record_id,lease_commence_year,TODAY_YEAR,NEXT_MONTH,lease_end_year,nb_years_left_from_next_year,nb_months_to_next_year
str,i64,i32,i32,i64,i64,i32
"""002-366463""",1979,2026,8,2078,51,4
"""002-366464""",1978,2026,8,2077,50,4
"""002-366465""",1978,2026,8,2077,50,4


In [13]:
# build the remaining_lease column
key_remaining_lease = (hdbdata_v2
                       .with_columns(remaining_lease = pl.format('{} years, {} months', col('nb_years_left_from_next_year'), col('nb_months_to_next_year')))
                       .select('record_id', 'remaining_lease')
)
print(key_remaining_lease)

shape: (92_543, 2)
┌────────────┬────────────────────┐
│ record_id  ┆ remaining_lease    │
│ ---        ┆ ---                │
│ str        ┆ str                │
╞════════════╪════════════════════╡
│ 002-366463 ┆ 51 years, 4 months │
│ 002-366464 ┆ 50 years, 4 months │
│ 002-366465 ┆ 50 years, 4 months │
│ 002-366466 ┆ 58 years, 4 months │
│ 002-366467 ┆ 58 years, 4 months │
│ …          ┆ …                  │
│ 004-459002 ┆ 72 years, 4 months │
│ 004-459003 ┆ 59 years, 4 months │
│ 004-459004 ┆ 64 years, 4 months │
│ 004-459005 ┆ 60 years, 4 months │
│ 004-459006 ┆ 59 years, 4 months │
└────────────┴────────────────────┘


In [14]:
# replace the remaining_lease column
hdbdata_replace_remanining_lease = (hdbdata_v1
                                     .drop('remaining_lease')
                                     .join(key_remaining_lease, on='record_id', how='left')
)

In [15]:
# validate
print('old remaining lease')
dc.showcol(hdbdata_v1, 'remaining_lease')

print('new remaining lease')
dc.showcol(hdbdata_replace_remanining_lease, 'remaining_lease')

hdbdata_v2 = hdbdata_replace_remanining_lease



old remaining lease
shape: (37_153, 1)
┌─────────────────┐
│ remaining_lease │
│ ---             │
│ i64             │
╞═════════════════╡
│ 70              │
│ 65              │
│ 64              │
│ 63              │
│ 64              │
│ …               │
│ 82              │
│ 69              │
│ 74              │
│ 70              │
│ 70              │
└─────────────────┘
new remaining lease
shape: (92_543, 1)
┌────────────────────┐
│ remaining_lease    │
│ ---                │
│ str                │
╞════════════════════╡
│ 51 years, 4 months │
│ 50 years, 4 months │
│ 50 years, 4 months │
│ 58 years, 4 months │
│ 58 years, 4 months │
│ …                  │
│ 72 years, 4 months │
│ 59 years, 4 months │
│ 64 years, 4 months │
│ 60 years, 4 months │
│ 59 years, 4 months │
└────────────────────┘


In [16]:
'''
REQUIREMENT: You may assume the composite key for the dataset is all columns except the resale price. If
all columns have the exact same value, except for the resale price (i.e. duplicated key), take
the higher price and discard the lower price into the “failed” dataset (if any).

My doubts: We assume if key is duplicated, but price is somehow exactly the same, they are not defined as duplicated key.
Logic can be modified if that was not intended.
'''

'\nREQUIREMENT: You may assume the composite key for the dataset is all columns except the resale price. If\nall columns have the exact same value, except for the resale price (i.e. duplicated key), take\nthe higher price and discard the lower price into the “failed” dataset (if any).\n\nMy doubts: We assume if key is duplicated, but price is somehow exactly the same, they are not defined as duplicated key.\nLogic can be modified if that was not intended.\n'

In [17]:
# composite key = every column except resale_price, excluding also record_id.
composite_key = [c for c in hdbdata_v2.columns
                       if c not in ('resale_price', 'record_id', 'tabling_version')]

dc.unicity(hdbdata_v2, composite_key)

victims will be returned in global as uvictims
victims are returned in global as uvictims.
multiplicity pcnt
90943
shape: (3, 3)
┌─────────┬───────┬──────────┐
│ nb_rows ┆ count ┆ pcnt     │
│ ---     ┆ ---   ┆ ---      │
│ u32     ┆ u32   ┆ f64      │
╞═════════╪═══════╪══════════╡
│ 1       ┆ 89379 ┆ 0.982802 │
│ 2       ┆ 1528  ┆ 0.016802 │
│ 3       ┆ 36    ┆ 0.000396 │
└─────────┴───────┴──────────┘
unicity ended


In [18]:
# # ════════════════════ SCRATCH / PROOF — UNCOMMENT TO VALIDATE, TURN OFF (RE-COMMENT IT) TO RUN PER FUNCTION ═══
# PROVE: 
# using price==max(price)
# If records in a composite-key group share the exact same max resale_price, both pass the filter and it will not unique into a record
# We will do a [unicity] check if there does exist many such cases.

hdbdata_v3_keyed = (hdbdata_v2
                 .with_columns(max_resale_price_in_key = col('resale_price').max().over(composite_key))
                 )

hdbdata_v3_deduped = (
    hdbdata_v3_keyed
    .filter(col('resale_price') == col('max_resale_price_in_key'))
    .drop('max_resale_price_in_key')
)

failed_duplicates = (
    hdbdata_v3_keyed
    .join(hdbdata_v3_deduped, on='record_id', how='anti')
    .drop('max_resale_price_in_key')
)

dc.unicity(hdbdata_v3_deduped, composite_key)
print('uniqueness is still not fulfilled')
# warning: multiplicity violated!
# victims are returned in global as uvictims.
# multiplicity pcnt
# 90944
# shape: (2, 3)
# ┌─────────┬───────┬─────────┐
# │ nb_rows ┆ count ┆ pcnt    │
# │ ---     ┆ ---   ┆ ---     │
# │ u32     ┆ u32   ┆ f64     │
# ╞═════════╪═══════╪═════════╡
# │ 1       ┆ 90673 ┆ 0.99702 │
# │ 2       ┆ 271   ┆ 0.00298 │ Tie in Resale Price
# └─────────┴───────┴─────────┘
# PROVE that data is still not unique in composite key
# # ══════════════════════════════════════════════════════════════════════════════════════════════════════════════

victims will be returned in global as uvictims
victims are returned in global as uvictims.
multiplicity pcnt
90943
shape: (2, 3)
┌─────────┬───────┬─────────┐
│ nb_rows ┆ count ┆ pcnt    │
│ ---     ┆ ---   ┆ ---     │
│ u32     ┆ u32   ┆ f64     │
╞═════════╪═══════╪═════════╡
│ 1       ┆ 90672 ┆ 0.99702 │
│ 2       ┆ 271   ┆ 0.00298 │
└─────────┴───────┴─────────┘
unicity ended
uniqueness is still not fulfilled


In [19]:

# Resolution: Alternate to tie breaker filter row_number == 1, over composite_key
# sort by resale_price descending (highest price first), then record_id ascending
# as a deterministic tiebreak, then keep only the first row encountered per composite-key
# group - this guarantees exactly one row survives per group, even on a genuine price tie.

hdbdata_v3_deduped = (
    hdbdata_v2
    .sort(['resale_price', 'record_id'], descending=[True, False])
    .unique(subset=composite_key, keep='first', maintain_order=True)
)

dc.unicity(hdbdata_v3_deduped, composite_key)
# passed

victims will be returned in global as uvictims
good!: multiplicity passed!:
unicity ended


In [20]:
hdbdata_v3_failed_composite = (
    hdbdata_v2
    .join(hdbdata_v3_deduped, on='record_id', how='anti')
)

In [21]:
print(f"kept: {hdbdata_v3_deduped.shape[0]:,} rows")
print(f"discarded to failed (lower-priced duplicates): {hdbdata_v3_failed_composite.shape[0]:,} rows")

kept: 90,943 rows
discarded to failed (lower-priced duplicates): 1,600 rows


In [22]:
#################
### DATA WRITE to Datalake
#################
if SAMPLED == 'on':
    print('on sampled data, no WRITE')
    print('break')
else:
    OUTPUT = hdbdata_v3_failed_composite
    OUTPUT.write_parquet('datalake/hdb/failed/hdbdata_duplicate_composite_discard', partition_by='tabling_version', mkdir=True)

    OUTPUT = hdbdata_v3_deduped
    OUTPUT.glimpse()
    OUTPUT.write_parquet('datalake/hdb/cleaned/hdbdata', partition_by=['tabling_version', 'month'], mkdir=True)
    print('written parquet')

Rows: 90943
Columns: 13
$ month               <str> '2016-12', '2016-09', '2016-08', '2016-11', '2016-11', '2014-10', '2015-11', '2016-06', '2016-08', '2016-01'
$ town                <str> 'KALLANG/WHAMPOA', 'CENTRAL AREA', 'KALLANG/WHAMPOA', 'CENTRAL AREA', 'CENTRAL AREA', 'BISHAN', 'CENTRAL AREA', 'CENTRAL AREA', 'CENTRAL AREA', 'CENTRAL AREA'
$ flat_type           <str> '3 ROOM', '5 ROOM', '5 ROOM', '5 ROOM', '5 ROOM', 'EXECUTIVE', '5 ROOM', '5 ROOM', '5 ROOM', '5 ROOM'
$ block               <str> '57', '1G', '8', '1B', '1D', '194', '1D', '1D', '1B', '1B'
$ street_name         <str> "JLN MA'MOR", 'CANTONMENT RD', 'BOON KENG RD', 'CANTONMENT RD', 'CANTONMENT RD', 'BISHAN ST 13', 'CANTONMENT RD', 'CANTONMENT RD', 'CANTONMENT RD', 'CANTONMENT RD'
$ storey_range        <str> '01 TO 03', '43 TO 45', '28 TO 30', '31 TO 33', '46 TO 48', '22 TO 24', '43 TO 45', '34 TO 36', '31 TO 33', '28 TO 30'
$ floor_area_sqm      <f64> 259.0, 106.0, 119.0, 106.0, 107.0, 150.0, 107.0, 107.0, 105.0, 108.0

written parquet
